In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load CSVs (they are in the working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

# Set timezone to UTC+8 (Asia/Shanghai) and create a datetime column for earliest timestamp calculations
tz = pytz.timezone('Asia/Shanghai')
for df in (df_metric, df_trace):
    # timestamps are in seconds (floored to minute). Create aware datetime in Asia/Shanghai.
    df['ts_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Aggregation helper: compute required stats on a Series
def agg_stats(s):
    return {
        'count': s.count(),
        'min': s.min(),
        'max': s.max(),
        'mean': s.mean(),
        'p95': np.percentile(s.dropna(), 95) if s.dropna().size>0 else np.nan,
        'p90': np.percentile(s.dropna(), 90) if s.dropna().size>0 else np.nan,
        'p15': np.percentile(s.dropna(), 15) if s.dropna().size>0 else np.nan,
        'p5' : np.percentile(s.dropna(), 5)  if s.dropna().size>0 else np.nan
    }

# --- Metrics aggregation (group by cmdb_id and kpi_name)
g_metric = df_metric.groupby(['cmdb_id', 'kpi_name'])
metrics_agg_full = g_metric['value'].apply(lambda s: pd.Series(agg_stats(s))).unstack(level=-1)
# The above produces a MultiIndex columns because apply returned a Series; flatten:
metrics_agg_full = metrics_agg_full.reset_index()
# Add earliest timestamp per group (using full series)
earliest_ts_metric = g_metric['ts_dt'].min().reset_index().rename(columns={'ts_dt': 'earliest_ts'})
metrics_agg_full = metrics_agg_full.merge(earliest_ts_metric, on=['cmdb_id','kpi_name'])

# Round numeric columns for compactness
num_cols = ['count','min','max','mean','p95','p90','p15','p5']
metrics_agg_full[num_cols] = metrics_agg_full[num_cols].round(6)

# Keep full aggregated table in memory as metrics_agg_full (do not print full table)

# --- Traces aggregation (group by cmdb_id and trace_name)
g_trace = df_trace.groupby(['cmdb_id', 'trace_name'])
traces_agg_full = g_trace['value'].apply(lambda s: pd.Series(agg_stats(s))).unstack(level=-1)
traces_agg_full = traces_agg_full.reset_index()
# Add earliest timestamp per group
earliest_ts_trace = g_trace['ts_dt'].min().reset_index().rename(columns={'ts_dt': 'earliest_ts'})
traces_agg_full = traces_agg_full.merge(earliest_ts_trace, on=['cmdb_id','trace_name'])
traces_agg_full[num_cols] = traces_agg_full[num_cols].round(6)

# Keep full aggregated table in memory as traces_agg_full (do not print full table)

# Prepare compact top-20 displays: sort by count desc, then by max desc, then head(20)
metrics_agg_top = metrics_agg_full.sort_values(['count','max'], ascending=[False, False]).head(20)[
    ['cmdb_id','kpi_name','count','earliest_ts','min','max','mean','p95','p90','p15','p5']
]

traces_agg_top = traces_agg_full.sort_values(['count','max'], ascending=[False, False]).head(20)[
    ['cmdb_id','trace_name','count','earliest_ts','min','max','mean','p95','p90','p15','p5']
]

# Display the two compact tables (variables shown per IPython rules)
metrics_agg_top, traces_agg_top

```
Out[1]:
```
```python
summary = (
    "I computed aggregated statistics (count, min, max, mean, P95, P90, P15, P5) for metric.csv and trace.csv "
    "grouped by (cmdb_id, kpi_name) and (cmdb_id, trace_name) respectively. The two compact top-20 tables were produced: "
    "metrics_agg_top and traces_agg_top. Key observations from those top rows:\n"
    "- Metrics: several db_* nodes show very high CPU_free_pct (means ~97–99%), indicating CPUs are largely idle. "
    "Memory utilization for db_007 is around 82% (stable). Disk I/O and network KPIs in the top rows are generally low. "
    "osb_001 service metrics (app.num, app.succee_num, app.avg_time) show substantial request counts (max ~990) and small average latencies.\n"
    "- Traces: docker_* containers show high row counts in JDBC/local traces (means ~1,100–1,800, max up to ~4,200), with per-container sample counts = 30. "
    "CSF trace counts are around ~450 mean. These trace values reflect workload/throughput rather than resource exhaustion.\n"
    "Overall: from these aggregates there is no immediate sign of CPU pressure or disk/network saturation. The prominent signal is high request/row counts in application traces and steady memory usage on some DB nodes; further investigation should focus on application-level throughput and DB query row volumes if performance issues are reported."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                  kpi_name  count               earliest_ts           min           max          mean           p95           p90           p15            p5
1783    osb_001                   app.num   61.0 2020-05-24 03:30:00+08:00  8.950000e+01  9.940000e+02  4.259918e+02  4.990000e+02  4.870000e+02  3.910000e+02  2.260000e+02
1784    osb_001            app.succee_num   61.0 2020-05-24 03:30:00+08:00  8.950000e+01  9.930000e+02  4.259754e+02  4.990000e+02  4.870000e+02  3.910000e+02  2.260000e+02
95       db_003              CPU_free_pct   61.0 2020-05-24 03:30:00+08:00  9.568578e+01  9.949408e+01  9.795908e+01  9.927238e+01  9.903636e+01  9.721599e+01  9.645711e+01
5        db_001              CPU_free_pct   61.0 2020-05-24 03:30:00+08:00  9.760310e+01  9.916050e+01  9.859768e+01  9.904596e+01  9.888429e+01  9.849355e+01  9.777083e+01
323      db_008              CPU_free_pct   61.0 2020-05-24 03:30:00+08:00  9.586549e+01  9.789644e+01  9.722217e+01  9.773130e+01  9.768717e+01  9.698717e+01  9.646981e+01
290      db_007             MEM_real_util   61.0 2020-05-24 03:30:00+08:00  8.230000e+01  8.233000e+01  8.231885e+01  8.233000e+01  8.233000e+01  8.231000e+01  8.230000e+01
1231     os_012              Disk_io_util   61.0 2020-05-24 03:30:00+08:00  0.000000e+00  3.000000e+00  7.428900e-01  2.000000e+00  1.500000e+00  8.246000e-03  0.000000e+00
1785    osb_001           app.succee_rate   61.0 2020-05-24 03:30:00+08:00  9.990000e-01  1.000000e+00  9.999840e-01  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
1517     os_017                Send_total   61.0 2020-05-24 03:30:00+08:00  8.652900e-02  5.425390e-01  2.305830e-01  2.639880e-01  2.573870e-01  2.102030e-01  1.738190e-01
1762     os_022  Outgoing_network_traffic   61.0 2020-05-24 03:30:00+08:00  2.718300e-02  1.261180e-01  5.927600e-02  6.781300e-02  6.682200e-02  5.413700e-02  4.822000e-02
1257     os_012      Processor_load_5_min   61.0 2020-05-24 03:30:00+08:00  5.000000e-03  3.000000e-02  1.368900e-02  2.833300e-02  2.000000e-02  8.333000e-03  6.667000e-03
1782    osb_001              app.avg_time   61.0 2020-05-24 03:30:00+08:00  4.410000e-04  7.928000e-03  8.200000e-04  1.742000e-03  8.600000e-04  4.730000e-04  4.520000e-04
1639     os_020              Disk_io_util   61.0 2020-05-24 03:30:00+08:00  0.000000e+00  5.557000e-03  5.440000e-04  5.524000e-03  0.000000e+00  0.000000e+00  0.000000e+00
1396     os_015  Incoming_network_traffic   61.0 2020-05-24 03:30:00+08:00  0.000000e+00  2.388000e-03  2.270000e-03  2.366000e-03  2.355000e-03  2.273000e-03  2.270000e-03
1058     os_008                Send_total   61.0 2020-05-24 03:30:00+08:00  5.130000e-04  6.870000e-04  5.800000e-04  6.840000e-04  6.800000e-04  5.210000e-04  5.140000e-04
956      os_006                Send_total   61.0 2020-05-24 03:30:00+08:00  3.030000e-04  4.960000e-04  3.710000e-04  4.770000e-04  4.710000e-04  3.130000e-04  3.110000e-04
977      os_007               Disk_rd_ios   61.0 2020-05-24 03:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1862  redis_004                 maxmemory   60.0 2020-05-24 03:30:00+08:00  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
1950  redis_008                 maxmemory   60.0 2020-05-24 03:30:00+08:00  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09
2038  redis_012                 maxmemory   60.0 2020-05-24 03:30:00+08:00  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09  8.000000e+09,         cmdb_id                  trace_name  count               earliest_ts    min     max         mean      p95     p90      p15       p5
99   docker_004   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  704.0  4214.0  1829.266667  2151.45  2082.6  1421.25  1055.45
15   docker_001   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  624.0  4204.0  1803.966667  2111.10  2051.6  1580.50   952.00
71   docker_003   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  704.0  3784.0  1793.666667  2234.95  2122.5  1553.45   944.80
43   docker_002   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  816.0  3716.0  1819.633333  2200.90  2103.8  1587.10   991.40
103  docker_004  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  616.0  3682.0  1599.733333  1880.20  1821.4  1242.85   923.35
19   docker_001  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  546.0  3668.0  1576.866667  1844.50  1793.4  1381.80   833.00
75   docker_003  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  616.0  3304.0  1568.933333  1955.10  1856.4  1358.70   826.70
47   docker_002  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  714.0  3248.0  1589.933333  1922.20  1838.2  1386.70   866.60
123  docker_005   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  438.0  2511.0  1127.533333  1295.40  1273.0   968.90   637.70
171  docker_008   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  464.0  2502.0  1131.366667  1287.15  1265.4  1067.80   635.30
139  docker_006   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  464.0  2479.0  1135.866667  1249.30  1229.6  1074.40   610.50
155  docker_007   trace.self.jdbc.row_count   30.0 2020-05-24 04:00:00+08:00  436.0  2438.0  1130.666667  1316.90  1261.2  1013.60   605.70
127  docker_005  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  258.0  1504.0   676.066667   779.55   768.2   579.40   382.60
143  docker_006  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  284.0  1490.0   681.933333   749.70   733.9   648.75   366.35
175  docker_008  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  280.0  1490.0   678.933333   774.70   763.4   640.85   382.40
159  docker_007  trace.self.local.row_count   30.0 2020-05-24 04:00:00+08:00  260.0  1474.0   678.266667   788.95   759.3   606.90   362.20
91   docker_004    trace.self.csf.row_count   30.0 2020-05-24 04:00:00+08:00  175.0  1052.0   457.066667   537.20   520.4   356.20   267.10
7    docker_001    trace.self.csf.row_count   30.0 2020-05-24 04:00:00+08:00  156.0  1048.0   450.533333   527.00   512.4   394.80   238.00
63   docker_003    trace.self.csf.row_count   30.0 2020-05-24 04:00:00+08:00  177.0   944.0   448.266667   558.60   530.4   388.20   238.00
35   docker_002    trace.self.csf.row_count   30.0 2020-05-24 04:00:00+08:00  204.0   928.0   454.266667   549.20   525.2   396.20   247.60)```
```

In [ ]:
# In[2]:

# Reuse existing variables: df_metric, metrics_agg_full, tz
import pandas as pd
# Define time window in UTC+8
start = pd.Timestamp("2020-05-24 04:00:00", tz=tz)
end   = pd.Timestamp("2020-05-24 04:30:00", tz=tz)

# Filter metric rows to the window (using ts_dt which is timezone-aware)
df_window = df_metric[(df_metric['ts_dt'] >= start) & (df_metric['ts_dt'] <= end)].copy()

# Prepare global thresholds (P95 and P5) from the precomputed full-aggregation table
thresholds = metrics_agg_full[['cmdb_id','kpi_name','p95','p5']].copy()

# Merge thresholds into windowed rows
df_window = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies per rule: high if value >= global_P95, low if value <= global_P5
df_window['anomaly_high'] = df_window['value'] >= df_window['p95']
df_window['anomaly_low']  = df_window['value'] <= df_window['p5']

# Keep full filtered anomaly rows in memory for follow-up (do not print). This includes only rows that are anomalies.
df_anom_rows = df_window[df_window['anomaly_high'] | df_window['anomaly_low']].copy()

# Aggregate anomalies by (cmdb_id, kpi_name)
if not df_anom_rows.empty:
    agg = df_anom_rows.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('ts_dt','min'),
        latest_anomaly_ts   = ('ts_dt','max'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        high_count = ('anomaly_high','sum'),
        low_count  = ('anomaly_low','sum')
    ).reset_index()
    # Compute prevalence percentages
    agg['high_pct'] = (agg['high_count'] / agg['anomaly_count'] * 100).round(1)
    agg['low_pct']  = (agg['low_count']  / agg['anomaly_count'] * 100).round(1)
    agg['anomaly_type_prevalence'] = agg.apply(
        lambda r: f"high:{r['high_pct']}%, low:{r['low_pct']}%", axis=1
    )
    # Keep full aggregated anomalies in memory
    anomalies_agg_full = agg.copy()
    # Prepare compact output with requested columns, sort by anomaly_count desc, limit 20
    anomalies_agg_top = agg.sort_values('anomaly_count', ascending=False)[
        ['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
         'max_value_in_window','min_value_in_window','anomaly_type_prevalence']
    ].head(20)
else:
    # No anomalies in window -> empty table with columns
    anomalies_agg_full = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
        'max_value_in_window','min_value_in_window','anomaly_type_prevalence'
    ])
    anomalies_agg_top = anomalies_agg_full

# Display the compact aggregated anomaly table (top 20)
anomalies_agg_top

```
Out[2]:
```
```python
summary = (
    "Anomaly summary for 2020-05-24 04:00–04:30+08 (aggregated by cmdb_id,kpi_name):\n"
    "- Total groups shown: 20 (top groups by anomaly count).\n"
    "- Top anomalies:\n"
    "  1) os_007 / Disk_rd_ios: 31 anomalies (entire window). Values = 0.00; flags show high:100% & low:100% — likely threshold equality or constant-value artifact.\n"
    "  2) osb_001 / app.succee_rate: 31 anomalies, mostly high (96.8%). Values ~0.999–1.000 (very high success rate samples flagged above P95).\n"
    "  3) db_010 / tnsping_result_time: 30 anomalies, predominately high (93.3%), values range 10.0–20.0 — indicates elevated tnsping latency in the window.\n"
    "- Other notable groups (each 30 samples): many hosts show ICMP_ping/Agent_ping anomalies (os_016, os_017, os_020, os_021, os_022) all flagged high=100% — consistent ping changes across nodes.\n"
    "- DB-related flags: db_011 (AWS/AIOS/Hang), db_009 (On_Off_State, Row_Lock) and db_008/db_010 AIOS/ACS show anomalies (mix of high/low prevalence depending on metric).\n"
    "- Some groups show both high and low prevalence at 100% (or both high & low present) — this suggests global P95/P5 thresholds may be equal or the metric is constant; review threshold computation for those KPIs.\n"
    "Recommended focus: investigate tnsping_result_time (db_010), app success-rate behavior (osb_001), and the disk read metric on os_007; also review ping/agent anomalies across the listed hosts and DB AIOS/Hang indicators."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window  min_value_in_window  anomaly_type_prevalence
925    os_007          Disk_rd_ios             31 2020-05-24 04:00:00+08:00 2020-05-24 04:30:00+08:00                 0.00                0.000  high:100.0%, low:100.0%
1723  osb_001      app.succee_rate             31 2020-05-24 04:00:00+08:00 2020-05-24 04:30:00+08:00                 1.00                0.999   high:96.8%, low:100.0%
441    db_010  tnsping_result_time             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                20.00               10.000     high:93.3%, low:6.7%
1441   os_017            ICMP_ping             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
1471   os_018           Agent_ping             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
1485   os_018          Disk_wr_ios             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 4.00                2.000    high:16.7%, low:83.3%
1379   os_016         Disk_io_util             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.00                0.000  high:100.0%, low:100.0%
1397   os_016      Memory_used_pct             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                26.44               26.420    high:13.3%, low:86.7%
1420   os_017           Agent_ping             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
444    db_011                  AWS             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.01                0.000    high:36.7%, low:63.3%
443    db_011                 AIOS             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.00                0.000  high:100.0%, low:100.0%
374    db_009         On_Off_State             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
452    db_011                 Hang             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
1691   os_022            ICMP_ping             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
381    db_009             Row_Lock             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.00                0.000  high:100.0%, low:100.0%
1647   os_021      Memory_used_pct             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                50.84               50.820    high:73.3%, low:26.7%
399    db_010                 AIOS             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.01                0.000   high:100.0%, low:96.7%
398    db_010                  ACS             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.02                0.010    high:10.0%, low:90.0%
1591   os_020            ICMP_ping             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 1.00                1.000  high:100.0%, low:100.0%
310    db_008                 AIOS             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                 0.01                0.000    high:10.0%, low:90.0%```
```

In [ ]:
# In[3]:

# Reuse existing variables: df_anom_rows, metrics_agg_full, tz, df_metric
import pandas as pd
import numpy as np

# If df_anom_rows doesn't exist or is empty, handle gracefully
try:
    df_anoms = df_anom_rows.copy()
except NameError:
    df_anoms = pd.DataFrame(columns=df_metric.columns.tolist() + ['p95','p5','anomaly_high','anomaly_low','ts_dt'])

# Ensure ts_dt is timezone-aware and sorted
df_anoms['ts_dt'] = pd.to_datetime(df_anoms['ts_dt']).dt.tz_convert(tz) if not df_anoms.empty else df_anoms.get('ts_dt')

if df_anoms.empty:
    sequences_full = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','seq_id','fault_start_ts','fault_end_ts','length_minutes',
        'max_value_in_sequence','min_value_in_sequence','high_count','low_count'
    ])
    sequences_top = sequences_full.head(0)
else:
    # For each group, identify contiguous sequences where gap >1 minute breaks sequence
    df_anoms = df_anoms.sort_values(['cmdb_id','kpi_name','ts_dt']).reset_index(drop=True)
    # Compute time diff in minutes within each group
    df_anoms['prev_ts'] = df_anoms.groupby(['cmdb_id','kpi_name'])['ts_dt'].shift(1)
    df_anoms['gap_min'] = (df_anoms['ts_dt'] - df_anoms['prev_ts']) / pd.Timedelta(minutes=1)
    # A new sequence starts when gap_min > 1 or prev_ts is NaT
    df_anoms['new_seq'] = (df_anoms['gap_min'].isna()) | (df_anoms['gap_min'] > 1.0)
    # Assign sequence ids per group
    df_anoms['seq_id'] = df_anoms.groupby(['cmdb_id','kpi_name'])['new_seq'].cumsum()

    # Aggregate per sequence
    seq_agg = df_anoms.groupby(['cmdb_id','kpi_name','seq_id']).agg(
        fault_start_ts = ('ts_dt','min'),
        fault_end_ts   = ('ts_dt','max'),
        length_minutes = ('ts_dt','size'),
        max_value_in_sequence = ('value','max'),
        min_value_in_sequence = ('value','min'),
        high_count = ('anomaly_high','sum'),
        low_count  = ('anomaly_low','sum')
    ).reset_index()

    # Attach global thresholds from metrics_agg_full (p95 and p5)
    thr = metrics_agg_full[['cmdb_id','kpi_name','p95','p5']].copy()
    seq_agg = seq_agg.merge(thr, on=['cmdb_id','kpi_name'], how='left')

    # Decide predominant anomaly type per sequence: 'high' if high_count >= low_count else 'low'
    seq_agg['threshold_type'] = np.where(seq_agg['high_count'] >= seq_agg['low_count'], 'P95', 'P5')
    seq_agg['threshold_value'] = np.where(seq_agg['threshold_type']=='P95', seq_agg['p95'], seq_agg['p5'])

    # Determine extreme_value: max_value for high, min_value for low
    seq_agg['extreme_value'] = np.where(seq_agg['threshold_type']=='P95',
                                        seq_agg['max_value_in_sequence'],
                                        seq_agg['min_value_in_sequence'])

    # Compute relative_breach_pct per formula; avoid division by zero via max(abs(threshold),1e-9)
    seq_agg['relative_breach_pct'] = (
        (seq_agg['extreme_value'] - seq_agg['threshold_value']).abs()
        / seq_agg['threshold_value'].abs().clip(lower=1e-9)
    ) * 100.0

    # Round relative_breach_pct for compactness
    seq_agg['relative_breach_pct'] = seq_agg['relative_breach_pct'].round(1)

    # Keep full sequence details in memory
    sequences_full = seq_agg.copy()

    # Filter sequences: length_minutes >= 2 AND relative_breach_pct > 5%
    seq_filtered = seq_agg[(seq_agg['length_minutes'] >= 2) & (seq_agg['relative_breach_pct'] > 5.0)].copy()

    # Prepare compact output columns and sorting
    if seq_filtered.empty:
        sequences_top = seq_filtered.head(0)[[
            'cmdb_id','kpi_name','fault_start_ts','fault_end_ts','length_minutes',
            'extreme_value','threshold_type','threshold_value','relative_breach_pct'
        ]]
    else:
        sequences_top = seq_filtered.sort_values(
            by=['length_minutes','relative_breach_pct'], ascending=[False, False]
        )[
            ['cmdb_id','kpi_name','fault_start_ts','fault_end_ts','length_minutes',
             'extreme_value','threshold_type','threshold_value','relative_breach_pct']
        ].head(20)

# Display the compact table (top 20)
sequences_top

```
Out[3]:
```
```python
summary = (
    "Fault identification summary (sequences >=2 minutes and relative breach >5%):\n"
    "- Many sustained sequences (most lasting the full 24–30 minute window) were detected across hosts and DBs.\n"
    "- Top sustained issues to investigate (examples from the compact table):\n"
    "  * db_012 / SctRead_Per_Sec: 30-min sustained high values (extreme 13.19). Threshold P95 recorded as 0 -> enormous relative breach. (Possible threshold artifact; verify P95 computation.)\n"
    "  * os_018 / Disk_rd_ios and os_010 / Disk_svctm: 30-min sustained disk metrics above P95 (thresholds recorded as 0 in our global stats), producing huge relative breach percentages. Investigate disk read/service time behavior on these hosts.\n"
    "  * db_011 / TPS_Per_Sec and several DB ACS/AIOS metrics (db_007, db_010, db_011, db_004, db_002): 30-min sustained deviations from P95 (small absolute values but notable percent breaches, e.g., ACS 0.03 vs P95 0.02 => 50%). Check DB throughput and these service-specific KPIs.\n"
    "  * db_012 and db_009 / tnsping_result_time: sustained elevated tnsping latencies (db_012: extreme 30 vs P95 20 => +50% breach; db_009: extreme 20 vs P95 10 => +100% breach). These are actionable signals of increased DB network/response latency.\n"
    "  * os_021 and os_022 / Disk_wr_ios: sustained low-side anomalies (values ~3 vs P5=4) with ~25% relative breach — indicates writes below expected baseline; check I/O patterns.\n"
    "- Important caveat: several metrics have global P95 or P5 equal to 0, producing extremely large relative_breach_pct values (division-by-small-threshold artifact). For those metrics, treat the percent numbers cautiously and re-evaluate thresholding logic (e.g., use a minimum threshold floor or absolute breach amount).\n"
    "Recommended next steps:\n"
    "  1) Immediately investigate tnsping_result_time for db_009 and db_012 (sustained elevated latencies).\n"
    "  2) Inspect disk service/read metrics on os_018, os_010 and other hosts with sustained anomalies.\n"
    "  3) Review DB ACS/AIOS/TPS metrics for the listed DB nodes for sustained unusual throughput/latency.\n"
    "  4) Recompute or sanity-check global thresholds where P95/P5 == 0 to avoid misleading relative breach percentages.\n"
    "These sequences are stored in memory for follow-up analysis (full sequence details available if needed)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name            fault_start_ts              fault_end_ts  length_minutes  extreme_value threshold_type  threshold_value  relative_breach_pct
2117     db_012            SctRead_Per_Sec 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30      13.190000            P95            0.000         1.319000e+12
5421     os_018                Disk_rd_ios 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       1.000000            P95            0.000         1.000000e+11
3988     os_010                 Disk_svctm 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.665000            P95            0.000         6.650000e+10
1998     db_011                TPS_Per_Sec 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.400000            P95            0.000         4.000000e+10
6063     os_022                 Disk_svctm 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.400000            P95            0.000         4.000000e+10
1111     db_007                       AIOS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.010000            P95            0.000         1.000000e+09
1675     db_010                       AIOS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.010000            P95            0.000         1.000000e+09
1844     db_011                        ACS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.020000            P95            0.010         1.000000e+02
250      db_002                        ACS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.030000            P95            0.020         5.000000e+01
577      db_004                        ACS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.030000            P95            0.020         5.000000e+01
1110     db_007                        ACS 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.030000            P95            0.020         5.000000e+01
2199     db_012        tnsping_result_time 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30      30.000000            P95           20.000         5.000000e+01
5919     os_021                Disk_wr_ios 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       3.000000             P5            4.000         2.500000e+01
6064     os_022                Disk_wr_ios 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       3.000000             P5            4.000         2.500000e+01
6291  redis_003  instantaneous_output_kbps 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00              30       0.070000            P95            0.060         1.670000e+01
1318     db_008                        AWS 2020-05-24 04:00:00+08:00 2020-05-24 04:27:00+08:00              28       0.030000            P95            0.020         5.000000e+01
3837     os_009                 Disk_await 2020-05-24 04:00:00+08:00 2020-05-24 04:23:00+08:00              24       0.660000            P95            0.000         6.600000e+10
1672     db_009        tnsping_result_time 2020-05-24 04:00:00+08:00 2020-05-24 04:23:00+08:00              24      20.000000            P95           10.000         1.000000e+02
12       db_001                        AWS 2020-05-24 04:02:00+08:00 2020-05-24 04:15:00+08:00              14       0.050000            P95            0.020         1.500000e+02
4413     os_012      Processor_load_15_min 2020-05-24 04:00:00+08:00 2020-05-24 04:09:00+08:00              10       0.016667            P95            0.015         1.110000e+01```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables: df_trace, traces_agg_full, tz
# Ensure tz is defined (UTC+8)
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Define incident window in UTC+8
start = pd.Timestamp("2020-05-24 04:00:00", tz=tz)
end   = pd.Timestamp("2020-05-24 04:30:00", tz=tz)

# Ensure df_trace has ts_dt (should from prior steps); if not, create it
if 'ts_dt' not in df_trace.columns:
    df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Filter trace rows to the window
df_trace_window = df_trace[(df_trace['ts_dt'] >= start) & (df_trace['ts_dt'] <= end)].copy()

# Prepare global thresholds from previously computed traces_agg_full (P95 and P5)
thresholds_trace = traces_agg_full[['cmdb_id','trace_name','p95','p5']].copy()

# Merge thresholds into windowed trace rows
df_trace_window = df_trace_window.merge(thresholds_trace, on=['cmdb_id','trace_name'], how='left')

# Mark anomalies: high if value >= global P95, low if value <= global P5
df_trace_window['anomaly_high'] = df_trace_window['value'] >= df_trace_window['p95']
df_trace_window['anomaly_low']  = df_trace_window['value'] <= df_trace_window['p5']

# Keep full filtered trace anomaly rows in memory for follow-up (do not print)
df_trace_anom_rows = df_trace_window[df_trace_window['anomaly_high'] | df_trace_window['anomaly_low']].copy()

# Aggregate anomalies by (cmdb_id, trace_name)
if not df_trace_anom_rows.empty:
    agg = df_trace_anom_rows.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count = ('value','size'),
        earliest_anomaly_ts = ('ts_dt','min'),
        latest_anomaly_ts   = ('ts_dt','max'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
        high_count = ('anomaly_high','sum'),
        low_count  = ('anomaly_low','sum')
    ).reset_index()
    # Determine predominant anomaly type
    agg['predominant_anomaly_type'] = np.where(agg['high_count'] >= agg['low_count'], 'high', 'low')
    # Select and round for compactness
    agg['max_value_in_window'] = agg['max_value_in_window'].round(6)
    agg['min_value_in_window'] = agg['min_value_in_window'].round(6)
    traces_anom_agg_full = agg.copy()  # keep full aggregated table in memory
    # Prepare compact top-20 output sorted by anomaly_count desc
    traces_anom_agg_top = agg.sort_values('anomaly_count', ascending=False)[
        ['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
         'max_value_in_window','min_value_in_window','predominant_anomaly_type']
    ].head(20)
else:
    # No anomalies found in window
    traces_anom_agg_full = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts',
        'max_value_in_window','min_value_in_window','predominant_anomaly_type'
    ])
    traces_anom_agg_top = traces_anom_agg_full

# Display the compact aggregated trace anomaly table (top 20)
traces_anom_agg_top

```
Out[4]:
```
Summary of trace anomaly aggregation for 2020-05-24 04:00–04:30+08:

- What I found:
  - Multiple trace error-rate metrics across docker containers (docker_001, docker_002, docker_003, etc.) show anomalies for every minute in the window (anomaly_count = 30, earliest = 04:00, latest = 04:29).
  - The affected trace names are mostly error-rate metrics (e.g., trace.self.jdbc.error_rate, trace.self.local.error_rate, trace.from_os_022.remoteprocess.error_rate, trace.to_docker_007.remoteprocess.error_rate).
  - Reported values in the window are all 0.0, yet they are flagged as "predominantly high" relative to the global P95 thresholds.

- Interpretation:
  - This pattern (value == 0.0 but marked as high) strongly suggests a thresholding artifact — global P95 for these traces is likely 0.0 or near-zero, causing equality/threshold-equality rows to be classified as anomalous.
  - It does not look like an actual increase in error rates (values are zero). Instead, either the global threshold computation or the threshold usage for equality needs review, or the metrics/time-series have degenerate distributions.

- Recommended next steps:
  1. Re-check the global P95/P5 thresholds for these trace metrics (ensure P95>0 where appropriate or apply a minimum floor to thresholds).
  2. Validate trace instrumentation and collection (confirm that 0.0 values are correct and not a collection artifact).
  3. If thresholds and instrumentation are correct, inspect application logs and container-level logs for docker_001/002/003 and the remoteprocess interactions named in traces to confirm there are no hidden errors despite zero error_rate readings.
  4. For correlation with system metrics, examine the stored full filtered trace anomaly rows (kept in memory) alongside metric anomalies previously computed.

If you want, I can recompute thresholds with a minimum floor or show the full filtered trace anomaly rows for targeted containers.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                    trace_name  anomaly_count       earliest_anomaly_ts         latest_anomaly_ts  max_value_in_window  min_value_in_window predominant_anomaly_type
2   docker_001    trace.from_os_022.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
6   docker_001                     trace.self.csf.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
14  docker_001                    trace.self.jdbc.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
10  docker_001               trace.self.flyremote.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
74  docker_003                   trace.self.local.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
78  docker_003  trace.to_docker_005.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
70  docker_003                    trace.self.jdbc.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
66  docker_003               trace.self.flyremote.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
34  docker_002                     trace.self.csf.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
38  docker_002               trace.self.flyremote.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
46  docker_002                   trace.self.local.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
42  docker_002                    trace.self.jdbc.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
26  docker_001  trace.to_docker_008.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
30  docker_002    trace.from_os_022.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
22  docker_001  trace.to_docker_007.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
18  docker_001                   trace.self.local.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
58  docker_003    trace.from_os_021.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
62  docker_003                     trace.self.csf.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
54  docker_002  trace.to_docker_008.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high
50  docker_002  trace.to_docker_007.remoteprocess.error_rate             30 2020-05-24 04:00:00+08:00 2020-05-24 04:29:00+08:00                  0.0                  0.0                     high```
```